# 🎬 Clipo - Auto-clip, auto-viral
### Alternatif Gratis, Tanpa Watermark, & Open-Source untuk `Opus Clip` dan `Vidyo.ai`
**Dibuat oleh: adewanggar** • [📸 Instagram: @alfansyahdr_](https://www.instagram.com/alfansyahdr_)

Clipo mengubah video panjang YouTube menjadi klip pendek siap viral (TikTok, Instagram Reels, YouTube Shorts) dengan:
- 🤖 **Pemotongan AI Otomatis**: Mendeteksi hook & momen paling menarik (Kie.ai GPT-Luna / Gemini / GPT-4 / LLM).
- 🗣️ **Subtitle Dinamis (Gaya Hormozi)**: Highlight kata per kata dengan transkripsi presisi tinggi (WhisperX).
- 🎥 **Face Tracking & Split Screen**: Deteksi wajah otomatis dan potong vertikal 9:16 atau split screen 2 orang.
- 🌍 **Terjemahan Otomatis**: Buat subtitle dalam berbagai bahasa secara instan.

---

### ⚠️ PENTING SEBELUM MEMULAI:
Pastikan **Akselerator GPU** sudah aktif:
1. Klik menu **Runtime** (di bar atas) ➔ **Change runtime type** (*Ubah jenis runtime*).
2. Pada opsi **Hardware accelerator**, pilih **T4 GPU**.
3. Klik **Save** (*Simpan*).

## 🛠️ Langkah 1: Instalasi & Setup Dependensi
Jalankan cell di bawah ini untuk menginstal semua library yang dibutuhkan (FFmpeg, PyTorch GPU, WhisperX, InsightFace, MediaPipe, dll). Proses ini memerlukan waktu sekitar **2-4 menit**.

In [ ]:
#@title 🛠️ Langkah 1: Jalankan Instalasi
import os
import subprocess
import shutil
from IPython.display import clear_output

# 1. Bersihkan instalasi lama jika ada
print("🧹 Membersihkan instalasi sebelumnya...")
%cd /content
if os.path.exists("ViralCutter"):
    shutil.rmtree("ViralCutter")

# 2. Clone repository Clipo
print("⏳ Mengunduh repository Clipo...")
!git clone https://github.com/adewanggar/clipo-autoclip.git ViralCutter
%cd /content/ViralCutter

# 3. Instal UV & Driver Sistem Linux
print("⏳ Menginstal UV package manager dan dependensi sistem (FFmpeg, CUDA, Xvfb)...")
subprocess.run(['pip', 'install', 'uv'], check=True)
subprocess.run('sudo apt update -y && sudo apt install -y libcudnn8 ffmpeg xvfb', shell=True, check=True)

# 4. Buat Virtual Environment
print("⏳ Membuat virtual environment (.venv)...")
subprocess.run(['uv', 'venv', '.venv'], check=True)

# 5. Instal Library & AI Models
print("⏳ Menginstal WhisperX, yt-dlp, dan dependensi dasar...")
cmds_fase_1 = [
    "uv pip install --python .venv git+https://github.com/m-bain/whisperx.git",
    "uv pip install --python .venv -r requirements.txt",
    "uv pip install --python .venv yt-dlp pytubefix requests",
    "uv pip install yt_dlp requests"
]
for cmd in cmds_fase_1:
    subprocess.run(cmd, shell=True, check=True)

print("⏳ Menginstal Google Generative AI, Pandas, dan Transformers yang kompatibel...")
extra_libs = [
    "uv pip install --python .venv google-generativeai",
    "uv pip install --python .venv pandas requests",
    "uv pip install --python .venv onnxruntime-gpu",
    "uv pip install --python .venv transformers==4.46.3 accelerate>=0.26.0"
]
for cmd in extra_libs:
    subprocess.run(cmd, shell=True, check=True)

print("🔨 Mengunci versi PyTorch stabil (2.3.1 + CUDA 12.1)...")
cmd_fix_torch = (
    "uv pip install --python .venv "
    "torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121 "
    "--index-url https://download.pytorch.org/whl/cu121"
)
subprocess.run(cmd_fix_torch, shell=True, check=True)

print("🔨 Mengunci versi Numpy...")
subprocess.run("uv pip install --python .venv 'numpy<2.0' setuptools==69.5.1", shell=True, check=True)

print("🔨 Mengonfigurasi InsightFace dan MediaPipe (Face Tracking)... ")
subprocess.run("uv pip install --python .venv insightface onnxruntime-gpu", shell=True, check=True)
subprocess.run("uv pip uninstall --python .venv mediapipe protobuf flatbuffers", shell=True)
subprocess.run("uv pip install --python .venv 'mediapipe>=0.10.0' 'protobuf>=3.20,<5.0' 'flatbuffers>=2.0'", shell=True, check=True)

# 6. Konfigurasi Virtual Display
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'

clear_output()
print("==========================================================")
print("✅ INSTALASI CLIPO BERHASIL & SELESAI!")
print("- PyTorch 2.3.1 (CUDA Aktif) : SIAP")
print("- WhisperX & Alignment      : SIAP")
print("- InsightFace / FaceTracker  : SIAP")
print("==========================================================")
print("👉 Lanjutkan dengan menjalankan Langkah 2 di bawah!")

## 🚀 Langkah 2: Konfigurasi API & Jalankan WebUI Clipo
Anda dapat memasukkan API Key di bawah (opsional, bisa juga diisi langsung di WebUI nanti).
Setelah dijalankan, klik tautan publik **`.gradio.live`** (contoh: `https://xxxx.gradio.live`) untuk membuka WebUI Clipo.

In [ ]:
#@title 🚀 Langkah 2: Konfigurasi API & Mulai WebUI { display-mode: "form" }
KIEAI_API_KEY = "" #@param {type:"string"}
GEMINI_API_KEY = "" #@param {type:"string"}
DEFAULT_AI_BACKEND = "kieai" #@param ["kieai", "gemini", "g4f"]

import os
import json

%cd /content/ViralCutter

# Update api_config.json jika ada API key yang dimasukkan
config_path = "/content/ViralCutter/api_config.json"
api_cfg = {}
if os.path.exists(config_path):
    try:
        with open(config_path, "r", encoding="utf-8") as f:
            api_cfg = json.load(f)
    except Exception:
        api_cfg = {}

if "kieai" not in api_cfg: api_cfg["kieai"] = {"model": "gpt-5-6-luna", "chunk_size": 20000, "reasoning_effort": "medium"}
if "gemini" not in api_cfg: api_cfg["gemini"] = {"model": "gemini-2.5-flash-lite-preview-09-2025", "chunk_size": 20000}
if "g4f" not in api_cfg: api_cfg["g4f"] = {"model": "gpt-4o-mini", "chunk_size": 2000}

api_cfg["selected_api"] = DEFAULT_AI_BACKEND
if KIEAI_API_KEY.strip():
    api_cfg["kieai"]["api_key"] = KIEAI_API_KEY.strip()
    print("🔑 API Key Kie.ai berhasil disimpan ke konfigurasi!")

if GEMINI_API_KEY.strip():
    api_cfg["gemini"]["api_key"] = GEMINI_API_KEY.strip()
    print("🔑 API Key Gemini berhasil disimpan ke konfigurasi!")

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(api_cfg, f, indent=4)

# Konfigurasi Virtual Display Headless
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'
os.environ['MPLBACKEND'] = 'Agg'

print("🚀 Menjalankan Clipo WebUI...")
print("⏳ Mohon tunggu beberapa detik hingga link public URL (gradio.live) muncul...")

# Jalankan webui dengan mode Colab
!/content/ViralCutter/.venv/bin/python webui/app.py --colab

## 💾 Langkah 3 (Opsional): Unduh Hasil Video Klip (ZIP)
Semua klip video yang telah selesai dipotong tersimpan di dalam folder `/content/ViralCutter/VIRALS`.
Jika Anda ingin mengunduh semua video yang telah dibuat ke komputer Anda sekaligus dalam format file `.zip`, jalankan cell di bawah ini:

In [ ]:
#@title 📥 Unduh Semua Video Hasil Render (.zip)
import os
import zipfile
from google.colab import files

output_dir = "/content/ViralCutter/VIRALS"
zip_filename = "/content/Clipo_Hasil_Klip.zip"

if os.path.exists(output_dir) and os.listdir(output_dir):
    print("📦 Mengompres folder VIRALS menjadi ZIP...")
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, filenames in os.walk(output_dir):
            for file in filenames:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, output_dir)
                zipf.write(file_path, arcname)
    print("⬇️ Memulai pengunduhan file ZIP ke komputer Anda...")
    files.download(zip_filename)
else:
    print("⚠️ Folder VIRALS masih kosong atau belum ada video yang diproses.")

## 👨‍💻 Pengembang & Kredit
- Dibuat / Dikembangkan oleh: **adewanggar** (Instagram: [**@alfansyahdr_**](https://www.instagram.com/alfansyahdr_))
- Terinspirasi oleh: [reels clips automator](https://github.com/eddieoz/reels-clips-automator) & [YoutubeVideoToAIPoweredShorts](https://github.com/Fitsbit/YoutubeVideoToAIPoweredShorts)
- Proyek ini bersifat Open Source (GPL-3.0 License)